# 📝 Bài tập: TF-IDF Thực hành
# Cho 3 tài liệu:
- **D1** = "AI is the future"  
- **D2** = "AI and machine learning"  
- **D3** = "machine learning is fun"  
---
# Yêu cầu:
1. **Tự cài đặt TF–IDF (không dùng thư viện có sẵn):**  
   - Viết hàm tính **TF**, **DF**, **IDF**, **TF–IDF**.  
   - In ra **bảng TF–IDF** cho toàn bộ từ vựng.
2. **So sánh kết quả** với thư viện `scikit-learn` (`TfidfVectorizer`).
3. **In ra từ quan trọng nhất** (có giá trị TF–IDF cao nhất) trong mỗi tài liệu.
4. **Mở rộng:**  
   - Thêm vào danh sách **2 tài liệu mới**, mỗi tài liệu chứa **ít nhất 10 từ**.  
   - Làm lại các bước **2** và **3** để so sánh kết quả.
---
📌 Gợi ý:
- Tách từ (tokenization) thủ công, chuẩn hóa chữ thường.
- Dùng `pandas.DataFrame` để hiển thị bảng TF-IDF đẹp hơn.
- So sánh trực quan kết quả tính thủ công với thư viện `scikit-learn`.

In [1]:
docs = [
    "AI is the future",
    "AI and machine learning",
    "machine learning is fun"
]

# Tách từ (tokenize) và chuẩn hóa
import re

tokenized_docs = [re.findall(r'\b\w+\b', doc.lower()) for doc in docs]
tokenized_docs


[['ai', 'is', 'the', 'future'],
 ['ai', 'and', 'machine', 'learning'],
 ['machine', 'learning', 'is', 'fun']]

In [2]:
# Cell 2: Xây dựng từ vựng
vocab = sorted(set(word for doc in tokenized_docs for word in doc))
vocab

['ai', 'and', 'fun', 'future', 'is', 'learning', 'machine', 'the']

In [3]:
# Cell 3: Hàm tính TF, DF, IDF

import math

def compute_tf(doc):
    tf = {}
    total_terms = len(doc)
    for term in doc:
        tf[term] = tf.get(term, 0) + 1
    for term in tf:
        tf[term] /= total_terms
    return tf

def compute_df(docs):
    df = {}
    for term in vocab:
        df[term] = sum(term in doc for doc in docs)
    return df

def compute_idf(docs):
    N = len(docs)
    df = compute_df(docs)
    idf = {term: math.log((N / df[term]), 10) for term in df}
    return idf

In [4]:
# Cell 4: Tính TF-IDF
idfs = compute_idf(tokenized_docs)

tfidf_matrix = []
for doc in tokenized_docs:
    tf = compute_tf(doc)
    tfidf = {term: tf.get(term, 0) * idfs[term] for term in vocab}
    tfidf_matrix.append(tfidf)

tfidf_matrix

[{'ai': 0.04402281476392031,
  'and': 0.0,
  'fun': 0.0,
  'future': 0.11928031367991561,
  'is': 0.04402281476392031,
  'learning': 0.0,
  'machine': 0.0,
  'the': 0.11928031367991561},
 {'ai': 0.04402281476392031,
  'and': 0.11928031367991561,
  'fun': 0.0,
  'future': 0.0,
  'is': 0.0,
  'learning': 0.04402281476392031,
  'machine': 0.04402281476392031,
  'the': 0.0},
 {'ai': 0.0,
  'and': 0.0,
  'fun': 0.11928031367991561,
  'future': 0.0,
  'is': 0.04402281476392031,
  'learning': 0.04402281476392031,
  'machine': 0.04402281476392031,
  'the': 0.0}]

In [ ]:
# Cell 5: In bảng TF-IDF dạng DataFrame
import pandas as pd

df_tfidf = pd.DataFrame(tfidf_matrix, columns=vocab)
df_tfidf.index = [f"D {i+1}" for i in range(len(docs))] # type: ignore
df_tfidf

,ai,and,fun,future,is,learning,machine,the
D 1,0.044023,0.00000,0.00000,0.11928,0.044023,0.000000,0.000000,0.11928
D 2,0.044023,0.11928,0.00000,0.00000,0.000000,0.044023,0.044023,0.00000
D 3,0.000000,0.00000,0.11928,0.00000,0.044023,0.044023,0.044023,0.00000


In [6]:
# Cell 6: So sánh với sklearn
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(docs)

pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out(), \
             index=[f"D{i+1}" for i in range(len(docs))]) # type: ignore

,ai,and,fun,future,is,learning,machine,the
D1,0.428046,0.000000,0.000000,0.562829,0.428046,0.000000,0.000000,0.562829
D2,0.459854,0.604652,0.000000,0.000000,0.000000,0.459854,0.459854,0.000000
D3,0.000000,0.000000,0.604652,0.000000,0.459854,0.459854,0.459854,0.000000


In [7]:
# Cell 7: In từ quan trọng nhất mỗi tài liệu
for i, row in df_tfidf.iterrows():
    top_word = row.idxmax()
    print(f"{i}: {top_word} ({row[top_word]:.4f})")

D 1: future (0.1193)
D 2: and (0.1193)
D 3: fun (0.1193)
